## 📚 벡터스토어 저장

In [2]:
import os
from dotenv import load_dotenv
import pickle
import faiss
import numpy as np
from langchain.embeddings import OpenAIEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document
import re
import os
from langchain_community.document_loaders import PDFPlumberLoader
from langchain_community.vectorstores import FAISS

from langchain.chat_models import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.schema.runnable import RunnableMap
from langchain.callbacks import get_openai_callback

In [3]:
load_dotenv()  # 환경변수 설정

True

In [5]:
openai_api_key = os.getenv("OPENAI_API_KEY")

if openai_api_key is None:
    raise ValueError("OPENAI_API_KEY가 설정되지 않았습니다.")

embeddings = OpenAIEmbeddings(openai_api_key=openai_api_key)

In [5]:
# 함수 정의
def remove_newlines_except_after_period(text):
    """마침표 다음의 줄바꿈을 제외한 모든 줄바꿈을 제거"""
    return re.sub(r'(?<!\.)(\n|\r\n)', ' ', text)


def save_vectorstore_with_documents(file_paths,vectorstore_path):
    """PDF 파일을 전처리하고 벡터스토어와 문서를 로컬에 저장하기"""
    all_docs=[]
    for file_path in file_paths:
        if os.path.exists(file_path):
            # 1.  문서 로딩
            loader = PDFPlumberLoader(file_path)
            docs = loader.load()            
            for doc in docs:
            # page_content에 대해 줄바꿈 처리 후 다시 할당
                doc.page_content = remove_newlines_except_after_period(doc.page_content)
            all_docs.extend(docs)
        else:
            print(f"파일을 찾을 수 없습니다: {file_path}")
    
    # 2. 문서 분할
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=50)
    split_documents = text_splitter.split_documents(all_docs)

    # 3. embedding 생성
    embeddings = OpenAIEmbeddings(openai_api_key=openai_api_key)

    # 4. 벡터스토어 생성하고 저장(retriever도 저장)
    vectorstore = FAISS.from_documents(documents=split_documents, embedding=embeddings)
    vectorstore.save_local(vectorstore_path)
    

    #5. 분할된 문서도 같이 저장
    document_paths = os.path.join(vectorstore_path, "documents.pkl")
    with open(document_paths, 'wb') as f:
        pickle.dump(split_documents, f)
    
    return vectorstore

In [ ]:
file_paths = ["./download/PBLN_000000000112667/(붙임) 2025년도 경남 유럽 시장개척단 참여기업 모집 공고문.pdf"] # 파일 경로만 각자 맞는대로 수정
vectorstore_path = "embeddings/faiss_index.index"  # 저장할 인덱스 경로
vectorstore = save_vectorstore_with_documents(file_paths,vectorstore_path)

## 🔍 Retrieve 시험해보기

In [8]:
vectorstore_path = "embeddings/faiss_index.index"  # 저장할 인덱스 경로

In [9]:
vectorstore = FAISS.load_local(vectorstore_path, OpenAIEmbeddings(openai_api_key=openai_api_key),allow_dangerous_deserialization=True)
query = "나 27살이고 카페 창업하고 싶은데 나한테 도움이 될만한 지원 사업이 있을까?"
query_embedding = OpenAIEmbeddings(openai_api_key=openai_api_key).embed_query(query)
query_embedding = np.array(query_embedding)
query_embedding = query_embedding.reshape(1, -1)

In [10]:
D, I = vectorstore.index.search(query_embedding, k=4)

In [11]:
D

array([[0.41456968, 0.42542192, 0.42795587, 0.4358346 ]], dtype=float32)

In [12]:
I

array([[ 5,  3,  0, 10]], dtype=int64)

In [13]:
document_paths = os.path.join(vectorstore_path, "documents.pkl")
with open(document_paths, 'rb') as f:
    documents = pickle.load(f)

In [14]:
# 4. 검색된 문서 내용 출력
for idx in I[0]:
    print(f"문서: {documents[idx].page_content}")

문서: 해외진출 ◦  회사  내부  정책,  외국어전담부서  배치,  영문화  작업  (카탈로그,  홈페이지),   노력도 수출을 위한 기반 구축 등  ◦ 프랑스 및 체코 등 유럽시장 진출 가능성, 수출의지 등을 잘 이해할 수 있도 신청사유 록 작성 상담희망 ◦ B2B 상담 희망 기업 또는 분야 작성 바이어 수출사업 ◦  주요  수출실적  및  현재  추진  중인  수출  현황(해외  바이어와  MOU체결,  현황 RFQ접수, 계약현황 등 구체적 기술) 향후  수출계획/ ◦ 본 사업 추진 이후 예상되는 수출계획 및 기대효과 등 작성 기대효과 「2025년도 경남 유럽 시장개척단」에 참여하고자 신청서를 제출합니다.
2025년 7월 일 신청기업 대표자 
문서: 시 소수점이 발생할 경우, 소수점 둘째 자리에서 반올림하여 산정 ○ 선정규모와 무관하게 평가점수가 평균 60점 이상인 경우만 선정 ○ 가점사항 (가점 중복인정 불가) - 해외시장개척단 신규 참가기업 - 경남도 및 중앙정부 지정 우수기업 □ 선정배제 대상 ○ ｢국가계약법｣ 제27조에 따른 부정당업자로 제재 중인 기업 ○ 국세 및 지방세를 체납 중인 기업 ○ 경남도 및 경남TP 추진 사업에 참여했다가 철회 또는 중도 포기한 기업 5. 신청기간 및 방법 □ 신청기간: 2025. 7. 22. ~ 2025. 8. 4. 15:00까지 □ 신청방법: 제출서류(서식 참조) 이메일 발송 □ 제출관련 문의: 한국원자력산업협회 박상언 대리(02-6953-2511, parks@kaif.or.kr) 6. 기타사항 □ 참가활동에 관한 상담일지, 참가결과보고서, 설문서를 작성하여 현장 활동 종료 시 제출해야 함 □ 참가이후 주관기관에서 계약(수출)실적 점검을 위한 자료 제출 요구 시 적극 협조하여야 함 □ 참가업체로 선정된 이후 중도 참가 포기 시 발생한 비용에 대해서는 해당 기업에서 전액 변상해야 함 □ 불성실 참가 기업 패널티 부여(3년간) □ 제출된 서류는 반환되지 않으며, 기재착오·서류제출 누락 및 제출된 서류 내용의 허위사실

## 🗨️ GPT-4o-mini 응답 생성해보기

In [15]:
# 벡터스토어 로딩
vectorstore = FAISS.load_local(
    vectorstore_path,
    embeddings,
    allow_dangerous_deserialization=True
)

# 리트리버 생성
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

In [ ]:
# LLM 설정
llm_answerer = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 답변용 프롬프트 템플릿
answer_prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 정부지원사업에 대해 잘 아는 컨설턴트야. 사용자 질문과 관련된 정보를 참고하여 이해하기 쉽게 설명해줘."),
    ("human", "질문: {question}\n\n참고할 수 있는 관련 공고 요약:\n{context}")
])

# 문서 리스트를 하나의 텍스트로 합치기
def combine_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 문서 검색 → 답변 생성 파이프라인
answer_chain = RunnableMap({
    "question": lambda x: x["question"],
    "context": lambda x: combine_docs(x["docs"])
}) | answer_prompt | llm_answerer

In [ ]:
question = "제가 하고있는 사업을 유럽에 진출시키고 싶은데요. 관련 지원 사업이 있을까요?"
retrieved_docs = retriever.invoke(question)

with get_openai_callback() as cb: 
    # 답변 생성
    response = answer_chain.invoke({
    "question": question,
    "docs": retrieved_docs
    })
    
    print("🧠 GPT-4o-mini 응답:\n")
    print(response.content)

    # ✅ 토큰/비용 출력
    print(f"총 사용된 토큰수: \t\t{cb.total_tokens}")
    print(f"프롬프트에 사용된 토큰수: \t{cb.prompt_tokens}")
    print(f"답변에 사용된 토큰수: \t{cb.completion_tokens}")
    print(f"호출에 청구된 금액(USD): \t${cb.total_cost:.6f}")

🧠 GPT-4o-mini 응답:

유럽 시장에 진출하고자 하는 사업에 대해 지원받을 수 있는 프로그램이 있습니다. 특히, 경상남도에서 시행하는 '2025년도 경남 유럽 시장개척단' 사업이 해당됩니다. 이 사업은 경남 지역의 원전기업을 대상으로 하며, 해외 판로 개척과 수출 성과 창출을 지원하는 프로그램입니다.

### 사업 개요
- **사업명**: 2025년도 해외 시장개척단 지원사업
- **사업기간**: 2025년 11월 2일 ~ 11월 7일 (5박 7일)
- **방문지역**: 프랑스, 체코
- **모집대상**: 경남도 내 소재 원전기업
- **모집규모**: 약 10개사 내외

### 지원 내용
- 왕복 항공료의 50% 지원 (1사 1인)
- 통역사 및 이동수단 제공
- B2B 상담 주선 및 산업 시찰 기회 제공

### 신청 방법
신청을 원하신다면, 아래의 정보를 포함한 신청서를 제출해야 합니다:
- 기업명, 사업자 등록번호, 연락처 등 기본 정보
- 수출 희망 품목 및 대상 국가
- 현재 추진 중인 수출 현황 및 향후 수출 계획
- 해외 규격 인증 보유 현황

### 기대 효과
이 프로그램에 참여함으로써, 경남 원전기업은 유럽 시장에 대한 네트워크를 구축하고, 실제 바이어와의 상담을 통해 수출 기회를 확대할 수 있습니다.

이 외에도 유럽 진출을 위한 다양한 지원 사업이 있을 수 있으니, 관련 기관에 문의하거나 추가 정보를 찾아보는 것도 좋습니다.
총 사용된 토큰수: 		1780
프롬프트에 사용된 토큰수: 	1398
답변에 사용된 토큰수: 	382
호출에 청구된 금액(USD): 	$0.000439


## 💸 GPT-3.5-turbo 로 시험해보기

In [19]:
# LLM 설정
llm_answerer = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

# 답변용 프롬프트 템플릿
answer_prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 정부지원사업에 대해 잘 아는 컨설턴트야. 사용자 질문과 관련된 정보를 참고하여 이해하기 쉽게 설명해줘."),
    ("human", "질문: {question}\n\n참고할 수 있는 관련 공고 요약:\n{context}")
])

# 문서 리스트를 하나의 텍스트로 합치기
def combine_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 문서 검색 → 답변 생성 파이프라인
answer_chain = RunnableMap({
    "question": lambda x: x["question"],
    "context": lambda x: combine_docs(x["docs"])
}) | answer_prompt | llm_answerer

In [21]:
question = "제가 하고있는 사업을 유럽에 진출시키고 싶은데요. 관련 지원 사업이 있을까요?"
retrieved_docs = retriever.invoke(question)

with get_openai_callback() as cb: 
    # 답변 생성
    response = answer_chain.invoke({
    "question": question,
    "docs": retrieved_docs
    })
    
    print("🧠 GPT-3.5-turbo 응답:\n")
    print(response.content)

    # ✅ 토큰/비용 출력
    print(f"총 사용된 토큰수: \t\t{cb.total_tokens}")
    print(f"프롬프트에 사용된 토큰수: \t{cb.prompt_tokens}")
    print(f"답변에 사용된 토큰수: \t{cb.completion_tokens}")
    print(f"호출에 청구된 금액(USD): \t${cb.total_cost:.6f}")

🧠 GPT-3.5-turbo 응답:

유럽에 사업을 진출시키고 싶은 경우, 한국에서 지원하는 정부사업을 통해 도움을 받을 수 있습니다. 예를 들어, 경남 원전기업의 해외판로 개척 및 수출성과 창출을 지원하는 '2025년도 경남 원전기업 유럽 시장개척단 사업'이 있습니다. 이 프로그램은 경남도 내 소재한 원전기업을 대상으로 하며, 프랑스와 체코를 방문하여 B2B 상담주선 및 산업시찰 등을 지원합니다.

이 프로그램은 왕복항공료의 50%를 지원하고, 기관당 통역사 및 이동수단 등을 제공합니다. 또한, 프랑스와 체코의 원전기업과의 상담 및 현지 산업시찰을 통해 해외 네트워크를 구축하고 수출 판로를 개척하는데 도움을 줍니다.

참여를 원하는 기업은 해당 기간에 맞춰 신청서를 작성하여 제출해야 합니다. 이 프로그램을 통해 유럽 시장에 진출하는데 도움을 받을 수 있을 것입니다.
총 사용된 토큰수: 		2521
프롬프트에 사용된 토큰수: 	2140
답변에 사용된 토큰수: 	381
호출에 청구된 금액(USD): 	$0.003972
